In [2]:
import numpy as np

# 1. 准备数据：固定矩阵 A (2x3) 和 原始输入变量 x (3x1)
A = np.array([[1, 0, 2],
              [0, 3, 1]])  # m=2, n=3
x = np.array([1.0, 2.0, 0.0]) # n=3

# 2. 定义讲义中的目标函数 f(z) = z^T z (即求向量元素的平方和)
def f(z):
    return z @ z

# 前向计算：z = Ax，然后求 f(z)
z = A @ x
print(f"z: {z}, z shape: {z.shape}")
print(f"f(Ax): {f(z)}\n")


z: [1. 6.], z shape: (2,)
f(Ax): 37.0



In [4]:
# ==========================================
# 感受核心冲突：∇f(Ax) 到底是什么？
# ==========================================

# 【解释 1】: ∇_z f(Ax) -> 认为是对中间变量 z 求导
# 根据讲义，公式为 2 * z，也就是 2 * (Ax)
grad_z = 2 * z
print("=== 解释 1：对中间变量 z 求梯度 ===")
print("公式算出的梯度 ∇_z f(Ax):", grad_z)
print("梯度的形状:", grad_z.shape, "-> (这是一个 m 维向量!)\n")

=== 解释 1：对中间变量 z 求梯度 ===
公式算出的梯度 ∇_z f(Ax): [ 2. 12.]
梯度的形状: (2,) -> (这是一个 m 维向量!)



In [6]:
# 【解释 2】: ∇_x f(Ax) -> 认为是对最原始的输入 x 求导
# 根据链式法则，公式为 2 * A^T * A * x
grad_x = 2 * A.T @ A @ x
print("=== 解释 2：对原始输入 x 求梯度 ===")
print("公式算出的梯度 ∇_x f(Ax):", grad_x)
print("梯度的形状:", grad_x.shape, "-> (这是一个 n 维向量!)\n")

=== 解释 2：对原始输入 x 求梯度 ===
公式算出的梯度 ∇_x f(Ax): [ 2. 36. 16.]
梯度的形状: (3,) -> (这是一个 n 维向量!)



In [11]:
print("=== 用纯代码“微调法”验证解释 2 ===")
# 我们不用公式，而是用最原始的笨办法：
# 把 x 的第 0 个元素微调 0.0001，看看最终的标量结果改变了多少倍
epsilon = 1e-4
x_tweaked = x.copy()
x_tweaked[0] += epsilon # 仅微调 x[0]

new_result = f(A @ x_tweaked)

# 变化率（导数） = (新结果 - 老结果) / 变动量
change_rate = (new_result - f(z)) / epsilon
print(f"手动微调 x[0] new result: {new_result}")
print(f"手动微调 x[0] 算出的变化率: {change_rate:.4f}")
print(f"前面代数公式 grad_x 里的第 0 个元素: {grad_x[0]}")
print("结论：两者完美吻合！")

=== 用纯代码“微调法”验证解释 2 ===
手动微调 x[0] new result: 37.00020001
手动微调 x[0] 算出的变化率: 2.0001
前面代数公式 grad_x 里的第 0 个元素: 2.0
结论：两者完美吻合！


In [12]:
# ==========================================
# 0. Prepare mathematical ingredients (fixed constants and variables)
# ==========================================
# Define the symmetric matrix A (2x2) as described in Section 4.3
A = np.array([[3.0, 1.0],
              [1.0, 4.0]])  # Off-diagonal elements are equal, satisfying A ∈ Sn

# Define the fixed coefficient vector b
b = np.array([5.0, 6.0])

# Define our input variable x 
# (We evaluate the calculus properties at the position x = [1.0, 2.0])
x = np.array([1.0, 2.0])

print("--- Preparation Complete ---")
print(f"Current variable position x: {x}\n")

--- Preparation Complete ---
Current variable position x: [1. 2.]



In [17]:
# ==========================================
# 1. Verifying Section 4.3 Formula 1: ∇_x (b^T x) = b
# ==========================================
# [Method A: Analytical Formula] According to the text, the result is simply vector b
grad_linear_formula = b

# [Method B: Numerical Finite Difference] 
# We tweak each element of x by epsilon and observe the change in function output
epsilon = 1e-5
grad_linear_numberic = np.zeros_like(x)

for i in range(len(x)):
    x_tweaked = x.copy()
    x_tweaked[i] += epsilon 

    # Linear function f(x) = b^T x, which is b @ x in NumPy
    output_original = b @ x
    output_tweaked = b @ x_tweaked

    # Calculate the rate of change (derivative)
    grad_linear_numberic[i] = (output_tweaked - output_original) / epsilon
    
print("=== [Verification 1] Gradient of Linear Function b^T x ===")
print("Analytical Formula Result:", grad_linear_formula)
print("Numerical Derivative Result:", grad_linear_numberic)
print("Conclusion: Both methods match perfectly!\n")

[0. 0.]
=== [Verification 1] Gradient of Linear Function b^T x ===
Analytical Formula Result: [5. 6.]
Numerical Derivative Result: [5. 6.]
Conclusion: Both methods match perfectly!



In [19]:
# ==========================================
# 2. Verifying Section 4.3 Formula 2: ∇_x (x^T A x) = 2Ax
# ==========================================
# [Method A: Analytical Formula] Derived on page 26 of the lecture notes: 2 * A * x
grad_quad_formula = 2 * A @ x

# [Method B: Numerical Finite Difference]
grad_quad_numeric = np.zeros_like(x)

for i in range(len(x)):
    x_tweaked = x.copy()
    x_tweaked[i] += epsilon
    # Quadratic function f(x) = x^T A x, which is x @ A @ x in NumPy
    output_original = x @ A @ x
    output_tweaked = x_tweaked @ A @ x_tweaked
    grad_quad_numeric[i] = (output_tweaked - output_original) / epsilon

print("=== [Verification 2] First-order Gradient of Quadratic Function x^T A x ===")
print("Analytical Formula Result (2Ax):", grad_quad_formula)
print("Numerical Derivative Result (▽f):", grad_quad_numeric)
print("Conclusion: Even with complex formulas, it is just rates of change under the hood!\n")

=== [Verification 2] First-order Gradient of Quadratic Function x^T A x ===
Analytical Formula Result (2Ax): [10. 18.]
Numerical Derivative Result (▽f): [10.00003 18.00004]
Conclusion: Even with complex formulas, it is just rates of change under the hood!



In [20]:
# ==========================================
# 3. Verifying Sections 4.2 & 4.3 Formula 3: Hessian Matrix ∇^2_x (x^T A x) = 2A
# ==========================================
# Section 4.2 explains that the j-th column of the Hessian matrix 
# is the partial derivative of the first-order gradient vector with respect to x_j.
# Hessian = Observe how the first-order gradient changes as x is slightly nudged.

# [Method A: Analytical Formula] The final elegant conclusion: 2 * A
hessian_formula = 2 * A

# [Method B: Numerical Finite Difference (Derivative of the Gradient)]
hessian_numeric = np.zeros((len(x), len(x)))

# Core Logic: We nudge x and observe how the entire first-order gradient vector changes
for j in range(len(x)):
    x_tweaked = x.copy()
    x_tweaked[j] += epsilon

    grad_original = 2 * A @ x
    grad_tweaked = 2 * A @ x_tweaked
    
    # Rate of change of gradient = (new_gradient - old_gradient) / epsilon
    # This forms the j-th column of the Hessian matrix!
    hessian_numeric[:, j] = (grad_tweaked - grad_original) / epsilon

print("=== [Verification 3] Second-order Hessian Matrix of x^T A x ===")
print("Analytical Hessian Formula (2A):\n", hessian_formula)
print("\nNumerical Hessian Result:\n", hessian_numeric)
print("\nConclusion: Look at that beautiful symmetric 2x2 table!")
print("The Hessian matrix is indeed the rate of change of the gradient vector!")

=== [Verification 3] Second-order Hessian Matrix of x^T A x ===
Analytical Hessian Formula (2A):
 [[6. 2.]
 [2. 8.]]

Numerical Hessian Result:
 [[6. 2.]
 [2. 8.]]

Conclusion: Look at that beautiful symmetric 2x2 table!
The Hessian matrix is indeed the rate of change of the gradient vector!
